# STAC data prep — GLHYMPS 2.0 (GeoParquet)

Converts the GLHYMPS 2.0 shapefile to a single **GeoParquet** file and writes the enriched metadata next to it.

- **Case:** vector data, one layer, 1.8 million polygons with 23 attribute columns, no time dimension (~3.7 GB .shp + 3 GB .dbf as shapefile). Source CRS: a projected cylindrical equal-area CRS (custom `Cylindrical_Equal_Area` .prj, no EPSG code).
- **Format:** one GeoParquet (see `DATA_FORMATS.md`, vector row; same approach as `18_hydrolakes` and `19_country_boundaries`). Rows are sorted along a Hilbert curve and a bbox covering column is written, so readers can filter spatially without loading everything.
- **OGC service:** not created. The GeoServer of the Global Coastal Atlas has no GeoParquet data store, and the shapefile is not published instead.
- **Metadata template:** `metadata/metadata_GLHYMPS.json`. It is incomplete (empty bbox, time, keywords, media type, collection id), so the missing fields are filled below.

Source: `P:\11209117-041-ipdc-egypt-swi\data\IntDeltaPlatform_QGA\Data_Vivian\GLHYMPS.shp`

**Running in docker:** the `P:` drive is not visible in the container, so run `docker\run_notebook_with_p_drive.ps1 -Notebook 22_glhymps.ipynb -Variable GLHYMPS` from the host. It copies the shapefile parts to a local mirror, executes this notebook, copies `stac\` back to `P:`, verifies the hashes and removes the mirror. The notebook needs about 20 GB of memory.

### Import packages

In [1]:
import json
import os
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

### Define drive paths

`DATA_VIVIAN` and `STAC_DATA_REPO` are set in the docker container; otherwise the Windows host paths are used.

In [2]:
VARIABLE = "GLHYMPS"
COLLECTION_ID = "glhymps"
PARQUET_NAME = "glhymps"

# DATA_VIVIAN is set in docker, where the P: drive is not mounted (a local mirror is used and stac/ is copied back to P:)
data_vivian = Path(os.environ.get("DATA_VIVIAN", r"P:\11209117-041-ipdc-egypt-swi\data\IntDeltaPlatform_QGA\Data_Vivian"))
stac_repo = Path(os.environ.get("STAC_DATA_REPO", r"C:\Ocean\Work\Projects\2025\STAC\Tools\Repositories\global-coastal-atlas\STAC\data"))

source_shp = data_vivian / f"{VARIABLE}.shp"
metadata_template_path = stac_repo / "metadata" / "metadata_GLHYMPS.json"

processed_data_dir = data_vivian / "stac" / COLLECTION_ID
parquet_dir = processed_data_dir / "parquet"
parquet_dir.mkdir(parents=True, exist_ok=True)

for p in (source_shp, metadata_template_path):
    if not p.is_file():
        raise FileNotFoundError(p)
print("Source:", source_shp)
print("Metadata template:", metadata_template_path)
print("Output dir:", processed_data_dir)

Source: P:\11209117-041-ipdc-egypt-swi\data\IntDeltaPlatform_QGA\Data_Vivian\GLHYMPS.shp
Metadata template: C:\Ocean\Work\Projects\2025\STAC\Tools\Repositories\global-coastal-atlas\STAC\data\metadata\metadata_GLHYMPS.json
Output dir: P:\11209117-041-ipdc-egypt-swi\data\IntDeltaPlatform_QGA\Data_Vivian\stac\glhymps


### Read raw data

Takes a few minutes and about 20 GB of memory.

In [3]:
gdf = gpd.read_file(source_shp, engine="pyogrio", encoding="utf-8")

empty = gdf.geometry.isna() | gdf.geometry.is_empty
print("rows:", len(gdf), "| crs:", gdf.crs.to_string() if gdf.crs else None)
print("geometry types:", gdf.geom_type.value_counts().to_dict())
print("OBJECTID_1 unique:", gdf["OBJECTID_1"].is_unique, "| null or empty geometries:", int(empty.sum()))
print("total bounds:", gdf.total_bounds.tolist())
gdf.drop(columns="geometry").head()

rows: 1803242 | crs: PROJCS["Cylindrical_Equal_Area",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0],UNIT["Degree",0.0174532925199433]],PROJECTION["Cylindrical_Equal_Area"],PARAMETER["standard_parallel_1",0],PARAMETER["central_meridian",0],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]
geometry types: {'Polygon': 1611574, 'MultiPolygon': 191668}
OBJECTID_1 unique: True | null or empty geometries: 0
total bounds: [-20037507.0672, -6363885.1193, 20037507.067199998, 6324895.855799999]


,OBJECTID_1,IDENTITY_,logK_Ice_x,logK_Ferr_,Porosity_x,K_stdev_x1,OBJECTID,Descriptio,XX,YY,...,Shape_Leng,GUM_K,Prmfrst,Shape_Le_1,Shape_Area,Transmissi,COUNT,AREA_1,MEAN,STD
0,900001,USA_VA5290,-1180,-1180,6,150,0,NaN,NaN,NaN,...,0.0,1,0,4071.184658,1.078709e+06,0.0,0,0.0,0.0,0.0
1,900002,USA_VA5313,-1650,-1650,12,170,0,NaN,NaN,NaN,...,0.0,1,0,15666.068657,2.424983e+06,0.0,0,0.0,0.0,0.0
2,900003,USA_VA5316,-1520,-1520,19,250,0,NaN,NaN,NaN,...,0.0,1,0,5531.310050,1.308683e+06,0.0,0,0.0,0.0,0.0
3,900004,USA_VA5320,-1180,-1180,6,150,0,NaN,NaN,NaN,...,0.0,1,0,13542.404828,2.151516e+06,0.0,0,0.0,0.0,0.0
4,900005,USA_VA5333,-1650,-1650,12,170,0,NaN,NaN,NaN,...,0.0,1,0,11060.458334,1.506812e+06,0.0,0,0.0,0.0,0.0


### Check the CRS

The shapefile is in a projected cylindrical equal-area CRS (custom `Cylindrical_Equal_Area` .prj, no EPSG code). The STAC catalog uses EPSG:4326, so the geometries are reprojected. Attributes are not touched: the `Shape_*` length/area fields keep the values of the original CRS.

In [4]:
print("CRS before:", gdf.crs.to_string() if gdf.crs else None)
if gdf.crs is None:
    gdf = gdf.set_crs(4326)
elif gdf.crs.to_epsg() != 4326:
    gdf = gdf.to_crs(4326)
print("CRS:", gdf.crs.to_string())
print("total bounds (EPSG:4326):", gdf.total_bounds.tolist())

CRS before: PROJCS["Cylindrical_Equal_Area",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0],UNIT["Degree",0.0174532925199433]],PROJECTION["Cylindrical_Equal_Area"],PARAMETER["standard_parallel_1",0],PARAMETER["central_meridian",0],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]
CRS: EPSG:4326
total bounds (EPSG:4326): [-179.99998854118687, -89.98525514923224, 179.99998854118684, 83.68252385852117]


### Make the metadata complete (dataset dependent)

Fill the fields that are empty in the template. Column descriptions follow the GLHYMPS 2.0 paper (Huscroft et al., 2018) and the GLHYMPS 1.0 documentation (Gleeson et al., 2014); please double-check the descriptions that are marked with `?` before publishing.

In [5]:
with open(metadata_template_path, encoding="utf-8") as f:
    metadata = json.load(f)

# Descriptions per attribute column (used as table:columns in the STAC item)
table_columns = {
    "OBJECTID_1": "Feature identifier",
    "IDENTITY_": "Polygon identifier (region code + number, e.g. USA_VA5290)",
    "logK_Ice_x": "Mean log10 permeability (m2) x 100, permafrost (ice) adjusted variant (?)",
    "logK_Ferr_": "Mean log10 permeability (m2) x 100, permafrost (Ferrians) adjusted variant (?)",
    "Porosity_x": "Porosity x 100 (percent)",
    "K_stdev_x1": "Standard deviation of log10 permeability x 100",
    "OBJECTID": "Identifier inherited from the source layer (0 where not available)",
    "Descriptio": "Lithology description from the source map",
    "XX": "Lithology/sediment class code 1 inherited from the source map (?)",
    "YY": "Lithology/sediment class code 2 inherited from the source map (?)",
    "ZZ": "Lithology/sediment class code 3 inherited from the source map (?)",
    "AA": "Lithology/sediment class code 4 inherited from the source map (?)",
    "DD": "Lithology/sediment class code 5 inherited from the source map (?)",
    "Shape_Leng": "Polygon perimeter in the original projected CRS (m, not updated after reprojection)",
    "GUM_K": "Flag: polygon permeability derived from the GUM unconsolidated sediment map (?)",
    "Prmfrst": "Permafrost flag (?)",
    "Shape_Le_1": "Polygon perimeter in the original projected CRS (m, not updated after reprojection)",
    "Shape_Area": "Polygon area in the original projected CRS (m2, not updated after reprojection)",
    "Transmissi": "Transmissivity (?; mostly 0)",
    "COUNT": "Zonal statistics helper field: number of raster cells (?; mostly 0)",
    "AREA_1": "Zonal statistics helper field: area (?; mostly 0)",
    "MEAN": "Zonal statistics helper field: mean value (?; mostly 0)",
    "STD": "Zonal statistics helper field: standard deviation (?; mostly 0)",
    "geometry": "Hydrogeological unit polygon (EPSG:4326)",
}
missing = set(gdf.columns) - set(table_columns)
assert not missing, f"Columns without description: {missing}"

west, south, east, north = (float(v) for v in gdf.total_bounds)
bbox = [max(west, -180.0), max(south, -90.0), min(east, 180.0), min(north, 90.0)]

# Extra info that the raw template does not contain
metadata.update({
    "TITLE_ABBREVIATION": "GLHYMPS 2.0",
    "HISTORY": [
        "GLHYMPS 2.0 (Huscroft et al., 2018), shapefile GLHYMPS.shp (projected cylindrical equal-area CRS)",
        "Reprojected to EPSG:4326 and converted to a single GeoParquet (zstd, rows sorted along a Hilbert curve, bbox covering column)",
        "Attributes are unchanged (the Shape_Leng, Shape_Le_1 and Shape_Area fields still hold the values of the original projected CRS)"
],
    "MEDIA_TYPE": "application/vnd.apache.parquet",
    "DATA_MODEL": "vector_geoparquet",
    "DIMENSIONS": [],
    "SPATIAL_EXTENT": [round(v, 6) for v in bbox],
    # Publication year; the dataset has no time dimension
    "TEMPORAL_EXTENT": ["2018-01-01T00:00:00Z", ""],
    "KEYWORDS": ["hydrogeology", "permeability", "porosity", "groundwater", "subsurface", "global"],
    "TAGS": ["hydrogeology", "permeability", "porosity", "polygons", "geoparquet"],
    "LONG_NAME": "Global subsurface permeability and porosity polygons",
    "UNITS": "",
    "COMMENT": (
        "Vector layer with one polygon per hydrogeological unit, with permeability, porosity and its uncertainty. "
        "Integer fields are scaled by 100 (e.g. Porosity_x = 12 means a porosity of 0.12; logK_* = -1180 means log10 "
        "k = -11.8 m2); the scaling and the fields marked with ? are to be confirmed against the GLHYMPS "
        "documentation. The temporal extent is the publication year; the layer is static."
    ),
    "AUTHOR": "Huscroft J, Gleeson T, Hartmann J, Börker J",
    "CRS": "EPSG:4326",
    "ITEM_BBOX_CRS": "EPSG:4326",
    "SPATIAL_RESOLUTION": None,
    "NODATA": None,
    "DATA_TYPE": "GeoParquet",
    "COLLECTION_ID": COLLECTION_ID,
    "WMS_DATASET": "",  # no OGC service: GeoServer has no GeoParquet data store
    "FEATURE_COUNT": int(len(gdf)),
    "TABLE_COLUMNS": table_columns,
})

print("SPATIAL_EXTENT:", metadata["SPATIAL_EXTENT"])
print("FEATURE_COUNT:", metadata["FEATURE_COUNT"])

SPATIAL_EXTENT: [-179.999989, -89.985255, 179.999989, 83.682524]
FEATURE_COUNT: 1803242


### Write data to GeoParquet

`zstd` compression, row groups of 100 000 features, rows ordered along a Hilbert curve so that nearby features share a row group, and a `bbox` covering column for spatial filtering. Null or empty geometries (if any) are put at the end, because they have no position on the curve.

In [6]:
valid = ~(gdf.geometry.isna() | gdf.geometry.is_empty).to_numpy()
hilbert = gdf.geometry[valid].hilbert_distance().to_numpy()
order = np.concatenate([np.flatnonzero(valid)[np.argsort(hilbert)], np.flatnonzero(~valid)])
gdf = gdf.iloc[order].reset_index(drop=True)

out_path = parquet_dir / f"{PARQUET_NAME}.parquet"
gdf.to_parquet(
    out_path,
    index=False,
    compression="zstd",
    row_group_size=100_000,
    write_covering_bbox=True,
)

metadata_output_path = processed_data_dir / f"metadata_{COLLECTION_ID}.json"
with open(metadata_output_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=4, ensure_ascii=False)

before_mb = sum(p.stat().st_size for p in source_shp.parent.glob(f"{VARIABLE}.*")) / 1e6
print(f"Wrote GeoParquet: {out_path} ({out_path.stat().st_size / 1e6:.0f} MB, shapefile parts {before_mb:.0f} MB)")
print("Wrote metadata:", metadata_output_path)

Wrote GeoParquet: P:\11209117-041-ipdc-egypt-swi\data\IntDeltaPlatform_QGA\Data_Vivian\stac\glhymps\parquet\glhymps.parquet (2931 MB, shapefile parts 7061 MB)
Wrote metadata: P:\11209117-041-ipdc-egypt-swi\data\IntDeltaPlatform_QGA\Data_Vivian\stac\glhymps\metadata_glhymps.json


### Verify the output

Same rows (in the same order), columns and values as the processed GeoDataFrame, EPSG:4326, readable as GeoParquet.

In [7]:
meta = pq.ParquetFile(out_path).metadata
print("row groups:", meta.num_row_groups, "| rows:", meta.num_rows)
assert meta.num_rows == len(gdf)

new = gpd.read_parquet(out_path)
assert new.crs.to_epsg() == 4326
assert list(new.columns.drop("bbox", errors="ignore")) == list(gdf.columns)

# Rows were written in the order of gdf, so they can be compared position by position
attrs = [c for c in gdf.columns if c != "geometry"]
pd.testing.assert_frame_equal(pd.DataFrame(new[attrs]), pd.DataFrame(gdf[attrs]))
same_geom = new.geometry.geom_equals_exact(gdf.geometry, tolerance=0) | (new.geometry.isna() & gdf.geometry.isna())
assert same_geom.all(), "Geometries changed"
print("OK: glhymps GeoParquet and metadata are ready for the STAC builder notebook.")

row groups: 19 | rows: 1803242
OK: glhymps GeoParquet and metadata are ready for the STAC builder notebook.
